# Data Quality with Great Expectations

This notebook demonstrates how to use Great Expectations to validate data in the data lake.

In [ ]:
import great_expectations as gx
from great_expectations.data_context import DataContext

## 1. Create a Data Context

In [ ]:
context = DataContext(context_root_dir='../great_expectations')

## 2. Connect to Data

In [ ]:
from pyspark.sql import SparkSession
import os
import boto3

# Create a Spark session
spark = SparkSession.builder \
    .appName("GreatExpectationsExample") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", os.environ.get('MINIO_ROOT_USER', 'minioadmin')) \
    .config("spark.hadoop.fs.s3a.secret.key", os.environ.get('MINIO_ROOT_PASSWORD', 'minioadmin123')) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# Create a sample CSV file
csv_data = 'firstname,middlename,lastname,dob,gender,salary\n' \
           'James,,Smith,36636,M,3000\n' \
           'Michael,Rose,,40288,M,4000\n' \
           'Robert,,Williams,42114,M,4000\n' \
           'Maria,Anne,Jones,39192,F,4000\n' \
           'Jen,Mary,Brown,,F,-1'
with open('sample_data.csv', 'w') as f:
    f.write(csv_data)

# Connect to MinIO and upload the sample data
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id=os.environ.get('MINIO_ROOT_USER', 'minioadmin'),
    aws_secret_access_key=os.environ.get('MINIO_ROOT_PASSWORD', 'minioadmin123')
)
s3_client.create_bucket(Bucket='data-quality-demo')
s3_client.upload_file('sample_data.csv', 'data-quality-demo', 'sample_data.csv')

# Read the data from MinIO
df = spark.read.csv("s3a://data-quality-demo/sample_data.csv", header=True)
df.show()

## 3. Create an Expectation Suite

In [ ]:
expectation_suite_name = "my_spark_suite"
suite = context.add_or_update_expectation_suite(expectation_suite_name=expectation_suite_name)

# Add expectations
suite.add_expectation(
    gx.expectationConfiguration.ExpectColumnValuesToNotBeNull(
        column="firstname"
    )
)

suite.add_expectation(
    gx.expectationConfiguration.ExpectColumnValuesToBeInSet(
        column="gender",
        value_set=["M", "F"]
    )
)

suite.add_expectation(
    gx.expectationConfiguration.ExpectColumnValuesToBeBetween(
        column="salary",
        min_value=1000,
        max_value=10000
    )
)

context.save_expectation_suite(expectation_suite=suite, expectation_suite_name=expectation_suite_name)

## 4. Validate the Data

In [ ]:
validator = context.sources.add_spark("my_spark_datasource").read_dataframe(df)
result = validator.validate(expectation_suite=suite)
print(result)

## 5. View the Validation Results

In [ ]:
context.build_data_docs()